# Gold — Lab Test Trends

Builds `mano_adb_outlook.gold.lab_test_trends` — one row per test type per calendar month with abnormal/critical rates and month-over-month trends.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ----- Aggregate lab_results by test_name, year, month -----
base = (
    spark.table("mano_adb_outlook.silver.lab_results")
    .withColumn("year", year("test_date"))
    .withColumn("month", month("test_date"))
    .groupBy("test_name", "year", "month")
    .agg(
        count("*").alias("total_tests"),
        count(when(col("result_status") == "Abnormal", True)).alias("abnormal_test_count"),
        count(when(col("result_status") == "Critical", True)).alias("critical_test_count"),
    )
)

# ----- Optional join to calendar for month_name / quarter_name -----
calendar = (
    spark.table("mano_adb_outlook.silver.Calendar")
    .select("date_key", "month_name", "quarter_name")
)

# Build a date_key for the 1st of each year-month to join calendar
df = (
    base.alias("b")
    .withColumn(
        "month_start_key",
        date_format(make_date(col("year"), col("month"), lit(1)), "yyyyMMdd").cast("int"),
    )
    .join(calendar.alias("c"), col("month_start_key") == col("c.date_key"), "left")
)

# ----- Derived rates -----
df = (
    df
    .withColumn("abnormal_rate_pct", round(col("abnormal_test_count") / col("total_tests") * 100, 2))
    .withColumn("critical_rate_pct", round(col("critical_test_count") / col("total_tests") * 100, 2))
)

# ----- Window: month-over-month trend -----
w = Window.partitionBy("test_name").orderBy("year", "month")
df = (
    df
    .withColumn("prev_month_abnormal_rate_pct", lag("abnormal_rate_pct").over(w))
    .withColumn(
        "abnormal_rate_change_pct",
        when(col("prev_month_abnormal_rate_pct").isNull(), None)
        .otherwise(round(col("abnormal_rate_pct") - col("prev_month_abnormal_rate_pct"), 2)),
    )
)

# ----- Select final columns -----
df_final = df.select(
    "test_name", "year", "month",
    col("c.month_name").alias("month_name"),
    col("c.quarter_name").alias("quarter_name"),
    "total_tests", "abnormal_test_count", "critical_test_count",
    col("abnormal_rate_pct").cast("decimal(5,2)"),
    col("critical_rate_pct").cast("decimal(5,2)"),
    col("prev_month_abnormal_rate_pct").cast("decimal(5,2)"),
    col("abnormal_rate_change_pct").cast("decimal(6,2)"),
)

# ----- Write to gold -----
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mano_adb_outlook.gold.lab_test_trends")



In [0]:
row_count = spark.table("mano_adb_outlook.gold.lab_test_trends").count()
dbutils.notebook.exit(str(row_count))